# Minor project on Kaggle: Gemma-2-2B-IT, stages S2–S4 (main study)

Runs the main study's data pipeline (`main-study/`, `harness/pipeline.py`) on free Kaggle T4s,
one **Save & Run All (Commit)** at a time. Scope and protocol: `main-study/prereg/S2.md`
(amendments of 2026-11-13) and `docs/DECISIONS.md` D21–D27.

| Order | Stage | Device | What |
|---|---|---|---|
| 0 | `estimate` | 2 × T4 | timed smoke test (64 GSM8K-train problems × 6 prompts) and the projected GPU-hours / disk; **stops everything if over budget** |
| 1 | `problems` | CPU | GSM-Symbolic `main`, study templates × 8 instances (split file from the repo) |
| 2 | `generate` | 2 × T4 | greedy traces: HF batched `generate`, KV cache, float32, eager attention, left padding, stop after the answer line, max 512 new tokens |
| 3 | `score` | CPU | answer, correctness, split, s0 (parser v4), read-out positions |
| 4 | `verify` | 2 × T4 | **gate**: teacher-forced TL logits must reproduce the HF tokens (≥ 0.99) on ≥ 200 traces stratified by template |
| 5 | `cache` | 2 × T4 | the same verified TL pass on all other traces + residual cache (float16 shards, all layers, pre-registered positions) |
| 6 | `sampled` | 2 × T4 | T = 0.7, k = 5, test-split failures only |
| 7 | `report` | CPU | provenance, cap fraction, disk use |

**Notebook settings (once):** *Accelerator* **GPU T4 x2**; *Internet* **on**; and the weights,
either *Add-ons → Secrets*: `HF_TOKEN` (a read token of an account that accepted the Gemma
licence), or *Add Input → Models*: `google/gemma-2`, Transformers, `gemma-2-2b-it`, version 2
(Kaggle's copy; no token needed). `MODEL_SOURCE` below picks one.

**Each commit:** attach the previous version's output (*Add Input → Your Work → this notebook*),
then *Save Version → Save & Run All (Commit)*. Finished stages are skipped; a stage cut off by
the session deadline resumes where it stopped; no trace is ever regenerated. Every GPU stage
runs its workers as subprocesses, so the GPUs are released as soon as a stage ends, and the
commit ends (and stops using quota) as soon as the last pending stage is done or the deadline
is near. Outputs live in `/kaggle/working/btp` (the code and the Python environment are kept
in `/tmp`, so they are not saved as output).

**Guards.** The study stages refuse to start while `prereg/S2.md` is a draft, and none start
if the estimate is over the 25 T4-hour budget. If the equivalence gate fails, the run stops;
set `FALLBACK_TO_TL = True` below and commit again to regenerate with TransformerLens, as
pre-registered.

In [ ]:
# 0. Settings for this commit
MODEL = 'gemma-2-2b-it'
RUN_ESTIMATE = True          # stage 0; skipped once reports/m1/estimate_<model>.json exists
RUN_STUDY = True             # stages 1-7 (refused while prereg/S2.md is a draft)
ONE_STAGE_PER_COMMIT = False # True: run only the next pending stage, then end the commit
FALLBACK_TO_TL = False       # only after a FAILED equivalence gate (prereg/S2.md A2)
REPO_URL = 'https://github.com/mehul24d/Reasoning-Failure-Diagnosis.git'
BRANCH = 'main'
# Model weights: 'hf' = Hugging Face with the HF_TOKEN secret; 'kaggle' = the Kaggle Models copy
# (Add Input -> Models -> google/gemma-2, Transformers, gemma-2-2b-it, version 2), no token
# needed; 'auto' = hf if the secret is attached, else kaggle.
MODEL_SOURCE = 'auto'
open('/tmp/btp_model_source', 'w').write(MODEL_SOURCE)

In [ ]:
# 1. Session state, written to /tmp/btp_session.py and re-run at the top of every cell, so a
#    single cell can be re-run after a kernel restart. The HF token comes from Kaggle secrets
#    and is never printed or written to a file.
SESSION = r'''
import os, sys, time, json, glob, shutil, subprocess
SESSION_HOURS = 11.5            # Kaggle stops a GPU session at 12 h
RESERVE_MIN = 20                # kept free at the end for the status files
WORK = '/kaggle/working/btp'    # BTP_DATA_ROOT: data/, cache/store/, reports/m1/ (saved as output)
REPO = '/tmp/Reasoning-Failure-Diagnosis'
MS = REPO + '/main-study'
VENV = '/tmp/venv'
os.makedirs(WORK, exist_ok=True)
if os.path.exists('/tmp/btp_offline'):
    os.environ.update(HF_HUB_OFFLINE='1', HF_DATASETS_OFFLINE='1')
os.environ.update(BTP_DATA_ROOT=WORK, TOKENIZERS_PARALLELISM='false',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                  PATH=VENV + '/bin:' + os.environ['PATH'], VIRTUAL_ENV=VENV, PYTHONPATH=MS)
if not os.path.exists('/tmp/btp_session_start'):
    open('/tmp/btp_session_start', 'w').write(str(time.time()))
T0 = float(open('/tmp/btp_session_start').read())
DEADLINE = T0 + SESSION_HOURS * 3600 - RESERVE_MIN * 60
SOURCE = open('/tmp/btp_model_source').read().strip() if os.path.exists('/tmp/btp_model_source') else 'auto'
KAGGLE_MODEL = sorted(glob.glob('/kaggle/input/gemma-2/transformers/gemma-2-2b-it/*/config.json'))
if not os.environ.get('HF_TOKEN') and SOURCE in ('auto', 'hf'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        if SOURCE == 'hf' or not KAGGLE_MODEL:
            raise RuntimeError('No HF_TOKEN secret (Add-ons -> Secrets) and no Kaggle Models input '
                               '(google/gemma-2 / transformers / gemma-2-2b-it)') from None
USE_KAGGLE_MODEL = SOURCE == 'kaggle' or (SOURCE == 'auto' and not os.environ.get('HF_TOKEN'))

def sh(cmd, ok=(0,)):
    # run a shell command, stream its output, return its exit code (raise unless it is in ok)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if rc not in ok:
        raise RuntimeError(f'command failed ({rc}): {cmd}')
    return rc
'''
open('/tmp/btp_session.py', 'w').write(SESSION)
exec(SESSION)
print('outputs ->', WORK, '| HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')), '| Kaggle model:', USE_KAGGLE_MODEL,
      '| stages must finish by', time.strftime('%H:%M', time.localtime(DEADLINE)))
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 2. Code (outside /kaggle/working, so it is not output) and the Python 3.13 environment of
#    main-study/requirements.txt (TransformerLens 2.x needs 3.13 with numpy 2; see that file).
if os.path.isdir(REPO):
    sh(f'git -C {REPO} pull --ff-only')
else:
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}')
sh(f'git -C {REPO} log --oneline -1')
if not os.path.exists(VENV + '/bin/python'):
    sh(f'{sys.executable} -m pip -q install uv')
    sh(f'uv venv --seed --python 3.13 {VENV}')
    sh(f'uv pip install --python {VENV}/bin/python -r {MS}/requirements.txt')
sh(f'''{VENV}/bin/python -c "import torch, transformers, transformer_lens; assert torch.cuda.is_available(), 'set Accelerator = GPU T4 x2'; print('torch', torch.__version__, 'transformers', transformers.__version__, '| GPUs', torch.cuda.device_count())"''')
sh(f'{VENV}/bin/python -m pip freeze > {WORK}/pip_freeze.txt')
if USE_KAGGLE_MODEL:
    # Put the Kaggle Models copy where Hugging Face looks for google/gemma-2-2b-it, cache the
    # public datasets while online, then go offline for the hub only. Nothing else changes.
    import hashlib
    src = os.path.dirname(KAGGLE_MODEL[-1])
    rev = hashlib.sha1(src.encode()).hexdigest()
    hub = os.path.expanduser('~/.cache/huggingface/hub/models--google--gemma-2-2b-it')
    os.makedirs(f'{hub}/snapshots/{rev}', exist_ok=True)
    os.makedirs(f'{hub}/refs', exist_ok=True)
    for f in os.listdir(src):
        dst = f'{hub}/snapshots/{rev}/{f}'
        if not os.path.exists(dst):
            os.symlink(os.path.join(src, f), dst)
    open(f'{hub}/refs/main', 'w').write(rev)
    sh(f'''{VENV}/bin/python -c "from datasets import load_dataset; load_dataset('openai/gsm8k', 'main', split='train'); load_dataset('apple/GSM-Symbolic', 'main', split='test'); print('datasets cached')"''')
    open('/tmp/btp_offline', 'w').write(src)
    print('model from Kaggle Models:', src)

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 3. Restore: copy the outputs of an attached earlier version into /kaggle/working/btp.
#    Existing files are never overwritten, so re-running is harmless; with nothing attached
#    this is a fresh start.
cands = [d for d in glob.glob('/kaggle/input/**/btp', recursive=True) if os.path.isdir(d)]
def nfiles(d):
    return sum(len(f) for _, _, f in os.walk(d))
restored = None
if cands:
    src = max(cands, key=nfiles)
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f)
            d = os.path.join(WORK, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True)
                shutil.copy2(s, d)
                n += 1
    restored = {'from': src, 'files_copied': n}
print('restored:', restored or 'nothing attached (fresh start)')
SETUP_MIN = (time.time() - T0) / 60
print(f'setup took {SETUP_MIN:.1f} min')

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 4. Offline unit tests (tiny random model, about a minute). A failure stops the commit.
sh(f'cd {MS} && {VENV}/bin/python -m pytest -q -x')
sh(f'''{VENV}/bin/python -c "from huggingface_hub import hf_hub_download; hf_hub_download('google/{MODEL}', 'config.json'); print('Gemma access OK')"''')

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 5. Stage 0: make estimate (timed smoke test + projection). Exit code 2 = over budget:
#    then nothing else runs, and the numbers go to the group to choose the next cut.
STATUS = {'estimate': 'skipped'}
est = f'{WORK}/reports/m1/estimate_{MODEL}.json'
if RUN_ESTIMATE and not os.path.exists(est):
    rc = sh(f'cd {MS} && {VENV}/bin/python -m harness.estimate --model {MODEL} '
            f'--setup-minutes {SETUP_MIN:.1f}', ok=(0, 2))
    STATUS['estimate'] = 'within budget' if rc == 0 else 'OVER BUDGET'
    shutil.rmtree(f'{WORK}/data/smoke/{MODEL}/store', ignore_errors=True)   # smoke only, not results
elif os.path.exists(est):
    p = json.load(open(est))['projection']
    STATUS['estimate'] = 'OVER BUDGET' if p['exceeds_budget'] else 'within budget'
if os.path.exists(est):
    print(open(est.replace('.json', '.md')).read())

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 6. Stages 1-7 in order, until the session deadline. Each GPU stage starts one worker per T4
#    and ends them when it is done; a stage cut off by the deadline resumes next commit.
STATUS.setdefault('estimate', 'unknown')
if FALLBACK_TO_TL:
    sh(f'cd {MS} && {VENV}/bin/python -m harness.pipeline fallback --model {MODEL}', ok=(0, 1))
if not RUN_STUDY:
    STATUS['study'] = 'not requested'
elif STATUS['estimate'] == 'OVER BUDGET':
    STATUS['study'] = 'not started: estimate over budget (propose a cut first)'
elif subprocess.run(f'cd {MS} && {VENV}/bin/python -c "import sys; from harness.pipeline import s2_is_binding; sys.exit(0 if s2_is_binding() else 1)"',
                    shell=True).returncode != 0:
    STATUS['study'] = 'not started: prereg/S2.md is still a draft (group sign-off first)'
else:
    stages = 'problems,generate,score,verify,cache,sampled,report'
    if ONE_STAGE_PER_COMMIT:
        st = json.loads(subprocess.run(f'cd {MS} && {VENV}/bin/python -m harness.pipeline status --model {MODEL}',
                                       shell=True, capture_output=True, text=True).stdout)
        pending = [s for s in stages.split(',') if st.get(s) != 'complete']
        stages = pending[0] if pending else ''
    if stages:
        rc = sh(f'cd {MS} && {VENV}/bin/python -m harness.pipeline run --model {MODEL} '
                f'--stages {stages} --deadline {DEADLINE:.0f}', ok=(0, 1, 3))
        STATUS['study'] = {0: 'stages complete', 3: 'partial (deadline): commit again', 1: 'FAILED (see above)'}[rc]
    else:
        STATUS['study'] = 'all stages complete'
print(json.dumps(STATUS, indent=1))

In [ ]:
exec(open('/tmp/btp_session.py').read())
# 7. End of the commit: stage status, disk use, and what to do next. Nothing keeps running
#    after this cell, so the session (and the GPU quota) ends here.
if 'STATUS' not in globals():
    STATUS = {}
st = subprocess.run(f'cd {MS} && {VENV}/bin/python -m harness.pipeline status --model {MODEL}',
                    shell=True, capture_output=True, text=True).stdout
du = subprocess.run(f'du -sh {WORK}/* 2>/dev/null', shell=True, capture_output=True, text=True).stdout
session = {'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'hours': round((time.time() - T0) / 3600, 2),
           'commit': subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                                    capture_output=True, text=True).stdout.strip(),
           'status': STATUS, 'stages': json.loads(st) if st.strip().startswith('{') else st, 'disk': du}
json.dump(session, open(f'{WORK}/session_status.json', 'w'), indent=2)
with open(f'{WORK}/sessions.jsonl', 'a') as f:
    f.write(json.dumps(session) + '\n')
shutil.make_archive('/kaggle/working/reports_m1', 'zip', f'{WORK}/reports')
print(json.dumps(session, indent=2))
done = isinstance(session['stages'], dict) and all(v == 'complete' for v in session['stages'].values())
print('\nAll stages complete: download reports_m1.zip (and the trace files) for the repo.' if done else
      "\nNot finished: attach this version's output as input and Save & Run All again.")